# ⚙️ Vector Databases & ANN Indexes: HNSW & Metadata Filtering

**My Objective:** I am transitioning the retrieval architecture from an ephemeral, in-memory Python list to a persistent, disk-backed Vector Database (ChromaDB). Enterprise systems require scalable search; performing brute-force exact cosine matching across millions of vectors is computationally impossible in real-time. I am implementing Approximate Nearest Neighbor (ANN) indexes, specifically HNSW, to trade a measurable fraction of `recall@k` accuracy for sub-linear search latency. Additionally, I am implementing hard metadata filtering at the database layer to guarantee strict tenant isolation.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "vectordbs") if (Path.cwd() / "vectordbs").exists() else str(Path.cwd()))

import numpy as np
import embedding_utils as EMBEDDING
import vdb_utils as VDB

emb = EMBEDDING.Embedder("bge-small")
client = VDB.open_db()                               
col = VDB.build_collection(client, emb)
print(col.count(), "rows in collection", repr(col.name))

## 1. Persistent Vector Storage (ChromaDB)
Keeping vectors in a native Python variable means the data disappears upon restart and requires full re-embedding. By leveraging ChromaDB, the system stores vectors on disk alongside their original text and metadata, enabling persistent, stateful retrieval.

In [ ]:
row = col.get(limit=1, include=["embeddings", "metadatas", "documents"])
print("id      :", row["ids"][0])
print("vector  :", np.round(row["embeddings"][0][:6], 3), "... (", len(row["embeddings"][0]), "numbers )")
print("metadata:", row["metadatas"][0])
print("text    :", row["documents"][0][:100], "...")

## 2. Embedded Semantic Search
The database engine directly handles the vector comparisons. The model encodes the incoming question, and ChromaDB natively executes the cosine distance sweep, sorting the results efficiently.

In [ ]:
for h in VDB.search(col, emb, "how long do refunds take"):
    m = h["meta"]
    print(f"{h['score']:.3f}  {m['title']} > {m['heading']:32} [{m['category']}, updated {m['updated']}]")

## 3. Hard Filtering & Tenant Isolation
Real enterprise systems must isolate data—serving "only this customer's documents" or "only current policies". ChromaDB's `where` clauses act as hard filters (e.g., enforcing a specific `category` or `updated` timestamp). Filters are absolute rules, fundamentally solving data leakage and access control vulnerabilities before the vector math even occurs.

In [ ]:
for where in [None, VDB.where_filter("payments"), VDB.where_filter("returns", "2026-06-01"), VDB.where_filter("products")]:
    hits = VDB.search(col, emb, "how long do refunds take", 3, where)
    print(where)
    for h in hits:
        print(f"    {h['score']:.3f}  {h['meta']['doc_id']:15} {h['meta']['category']:9} {h['meta']['updated']}")

## 4. Upserting & Mutable State
When a policy changes, we do not rebuild the entire index. We re-embed only the modified document chunk and perform an `upsert` (update if the ID exists, insert if new) to instantly mutate the persistent database state without a full rebuild.

In [ ]:
q = "can I pay in instalments with my credit card"
print("before:", [h["meta"]["heading"] for h in VDB.search(col, emb, q, 2)])

row = col.get(where={"doc_id": "payments"}, limit=1, include=["metadatas", "documents"])
new_text = row["documents"][0] + " NEW: no-cost EMI in 3, 6 or 9 monthly instalments on all major credit cards."
col.upsert(ids=row["ids"], documents=[new_text], metadatas=row["metadatas"], embeddings=[emb.embed([new_text])[0].tolist()])

print("after :", [h["meta"]["heading"] for h in VDB.search(col, emb, q, 2)])
col = VDB.build_collection(client, emb)              # Restores the original text

## 5. The Limits of Exact Search (Brute Force)
Brute-force exact search compares the query against every single vector in the system. As demonstrated below, search latency and memory footprint scale perfectly linearly. A production database of 50 million products cannot sustain linear scaling per query.

In [ ]:
import time
for n in (10_000, 50_000, 200_000):
    x, qs = VDB.clustered_vectors(n, n_queries=20)
    t0 = time.perf_counter()
    for qv in qs:
        _ = x @ qv
    print(f"{n:>8,} vectors: {(time.perf_counter() - t0) * 1000 / len(qs):6.2f} ms per question, {x.nbytes / 1e6:5.0f} MB of vectors")

## 6. Approximate Nearest Neighbor (HNSW) Benchmarking
To break the linear scale barrier, I benchmarked the Hierarchical Navigable Small World (HNSW) ANN index. HNSW uses hierarchical layers of interconnected graphs to skip the vast majority of vectors. By tuning `ef_search` (candidate pool) and `M` (maximum neighbor links), we can precisely engineer the trade-off between search latency (`ms/query`) and retrieval accuracy (`recall@10`).

In [ ]:
x, qs = VDB.clustered_vectors(20_000)
exact = VDB.exact_top_k(x, qs, 10)
ann = VDB.open_db(None)                               
print(f"{'setting':9} {'ef_search':>9} {'M':>3} {'recall@10':>9} {'ms/query':>9}")
for preset, p in VDB.HNSW_PRESETS.items():
    r = VDB.hnsw_run(ann, x, qs, preset)
    print(f"{preset:9} {p['ef_search']:>9} {p['max_neighbors']:>3} {VDB.recall_at_k(exact, r['ids']):>9.2f} {r['ms_per_query']:>9.2f}")